# 🩺 Diabetes Insulin Dose Predictor — Full ML Pipeline
> **Advanced machine learning project** | Random Forest · Gradient Boosting · XGBoost · SHAP explainability  
> *Educational purposes only — not for real medical decisions*

---
## 📋 Notebook Overview

| # | Section | Models / Techniques |
|---|---------|-------------------|
| 1 | Data Generation & EDA | Synthetic physiological simulation |
| 2 | Feature Engineering | Interaction terms, time features |
| 3 | Model 1 — Insulin Dose | RF · GB · XGBoost · comparison |
| 4 | Model 2 — Post-Meal BG | Gradient Boosting Regressor |
| 5 | Model 3 — Hypo Risk | RF Classifier · ROC · Confusion matrix |
| 6 | Hyperparameter Tuning | GridSearchCV / RandomizedSearchCV |
| 7 | SHAP Explainability | Feature importance + force plots |
| 8 | Save & Deploy | joblib serialization |

## 1️⃣ Imports & Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                               RandomForestClassifier, GradientBoostingClassifier)
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.model_selection import (train_test_split, cross_val_score,
                                      GridSearchCV, RandomizedSearchCV,
                                      StratifiedKFold, KFold)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                              accuracy_score, classification_report,
                              confusion_matrix, roc_auc_score, roc_curve)
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance
import joblib, os

# Try importing XGBoost (install with: pip install xgboost)
try:
    from xgboost import XGBRegressor, XGBClassifier
    HAS_XGB = True
    print("✅ XGBoost available")
except ImportError:
    HAS_XGB = False
    print("⚠️  XGBoost not installed — skipping XGB models (pip install xgboost)")

# Try importing SHAP (install with: pip install shap)
try:
    import shap
    HAS_SHAP = True
    print("✅ SHAP available")
except ImportError:
    HAS_SHAP = False
    print("⚠️  SHAP not installed — skipping explainability section (pip install shap)")

np.random.seed(42)
plt.style.use('dark_background')
ACCENT = '#58d5a0'
WARN   = '#f87171'
GOLD   = '#f7c948'
COLORS = [ACCENT, GOLD, WARN, '#a78bfa', '#60a5fa']
print("\n🔧 Setup complete.")

## 2️⃣ Synthetic Data Generation

We simulate a cohort of diabetic patients with realistic physiological relationships.  
Replace this section with a real dataset (e.g. OhioT1DM, Pima, UCI Diabetes).

### Feature Design
| Feature | Range | Notes |
|---------|-------|-------|
| `carbs_g` | 10–130 g | Primary driver of insulin dose |
| `current_bg` | 50–400 mg/dL | Pre-meal blood glucose |
| `time_of_day` | 0–23 | Dawn phenomenon affects insulin sensitivity |
| `activity_level` | 0–3 | Each level reduces insulin need ~8% |
| `weight_kg` | 45–130 kg | Used for ICR/CF estimation |
| `icr` | 5–20 g/unit | Patient-specific insulin-to-carb ratio |
| `cf` | 20–100 mg/dL/unit | Patient-specific correction factor |
| `target_bg` | 80–130 mg/dL | Physician-set BG target |

In [ ]:
def generate_dataset(n_samples: int = 3000) -> pd.DataFrame:
    """
    Simulate realistic diabetes patient meal records.
    
    Physiological model:
      meal_dose       = (carbs / ICR) * activity_factor
      correction_dose = max(current_bg - target_bg, 0) / CF
      total_dose      = meal_dose + correction_dose + N(0, 0.3)
      
      bg_after_2h     = current_bg + carbs*4 - dose*CF - activity_drop + N(0,15)
      hypo_risk       = 0 (Low) / 1 (Medium) / 2 (High) based on bg_after_2h
    """
    rng = np.random.default_rng(42)

    # ── Patient-specific parameters (simulate multi-patient cohort)
    weight_kg  = rng.normal(75, 15, n_samples).clip(45, 130)
    icr        = rng.normal(10, 3,  n_samples).clip(5, 20)
    cf         = rng.normal(50, 12, n_samples).clip(20, 100)
    target_bg  = rng.normal(100, 10, n_samples).clip(80, 130)

    # ── Meal features
    carbs_g        = rng.uniform(10, 130, n_samples)
    current_bg     = rng.normal(140, 55, n_samples).clip(50, 400)
    time_of_day    = rng.integers(0, 24, n_samples)
    activity_level = rng.choice([0, 1, 2, 3], n_samples, p=[0.35, 0.30, 0.25, 0.10])

    # ── Dawn phenomenon: BG harder to control in early morning (4–9am)
    dawn_factor = np.where((time_of_day >= 4) & (time_of_day <= 9), 1.15, 1.0)

    # ── Activity reduces insulin sensitivity
    activity_factor = 1 - (activity_level * 0.08)

    # ── Ground-truth dose calculation
    meal_dose       = (carbs_g / icr) * activity_factor * dawn_factor
    correction_dose = np.maximum((current_bg - target_bg) / cf, 0)
    insulin_dose    = (meal_dose + correction_dose + rng.normal(0, 0.3, n_samples)).clip(0, 35)

    # ── Post-meal BG (simplified PK/PD model)
    bg_rise       = carbs_g * 4.2
    bg_drop       = insulin_dose * cf
    activity_drop = activity_level * 18 * (1 + rng.normal(0, 0.1, n_samples))
    dawn_bg_add   = np.where((time_of_day >= 4) & (time_of_day <= 9), 20, 0)
    bg_after_2h   = (current_bg + bg_rise - bg_drop - activity_drop + dawn_bg_add
                     + rng.normal(0, 18, n_samples)).clip(35, 500)

    # ── Hypoglycemia risk (multi-class classification target)
    hypo_risk = np.where(bg_after_2h < 70,  2,          # High
                np.where(bg_after_2h < 100, 1, 0))       # Medium / Low

    return pd.DataFrame({
        'carbs_g':        carbs_g.round(1),
        'current_bg':     current_bg.round(1),
        'time_of_day':    time_of_day,
        'activity_level': activity_level,
        'weight_kg':      weight_kg.round(1),
        'icr':            icr.round(1),
        'cf':             cf.round(1),
        'target_bg':      target_bg.round(1),
        'insulin_dose':   insulin_dose.round(2),
        'bg_after_2h':    bg_after_2h.round(1),
        'hypo_risk':      hypo_risk,
    })

df = generate_dataset(3000)
print(f"Dataset shape: {df.shape}")
print(f"\nFirst 5 rows:")
df.head()

### 📊 Exploratory Data Analysis

In [ ]:
print("=== Descriptive Statistics ===")
print(df.describe().round(2))
print("\n=== Target Distributions ===")
print(f"insulin_dose  : mean={df.insulin_dose.mean():.2f}u  std={df.insulin_dose.std():.2f}u  range=[{df.insulin_dose.min():.1f}, {df.insulin_dose.max():.1f}]")
print(f"bg_after_2h   : mean={df.bg_after_2h.mean():.1f}  std={df.bg_after_2h.std():.1f}  range=[{df.bg_after_2h.min():.0f}, {df.bg_after_2h.max():.0f}]")
print(f"hypo_risk     : {dict(zip(['Low','Medium','High'], df.hypo_risk.value_counts().sort_index().values))}")

In [ ]:
fig = plt.figure(figsize=(16, 10))
fig.patch.set_facecolor('#0d1117')

gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# 1. Insulin dose distribution
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(df['insulin_dose'], bins=50, color=ACCENT, alpha=0.85, edgecolor='none')
ax1.set_title('Insulin Dose Distribution', color=ACCENT, fontsize=11, fontweight='bold')
ax1.set_xlabel('Units', color='#8b949e'); ax1.set_ylabel('Count', color='#8b949e')
ax1.tick_params(colors='#8b949e'); ax1.set_facecolor('#161b22')

# 2. BG after 2h distribution
ax2 = fig.add_subplot(gs[0, 1])
ax2.hist(df['bg_after_2h'], bins=50, color=GOLD, alpha=0.85, edgecolor='none')
ax2.axvline(70,  color=WARN,  linestyle='--', linewidth=1.5, label='Hypo <70')
ax2.axvline(180, color=GOLD,  linestyle='--', linewidth=1.5, label='High >180')
ax2.set_title('Post-Meal BG Distribution', color=GOLD, fontsize=11, fontweight='bold')
ax2.set_xlabel('mg/dL', color='#8b949e'); ax2.legend(fontsize=8)
ax2.tick_params(colors='#8b949e'); ax2.set_facecolor('#161b22')

# 3. Hypo risk class balance
ax3 = fig.add_subplot(gs[0, 2])
counts = df['hypo_risk'].value_counts().sort_index()
bars = ax3.bar(['Low', 'Medium', 'High'], counts.values, color=[ACCENT, GOLD, WARN], alpha=0.85)
for bar, val in zip(bars, counts.values):
    ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+20, str(val),
             ha='center', va='bottom', color='white', fontsize=10)
ax3.set_title('Hypoglycemia Risk Classes', color=WARN, fontsize=11, fontweight='bold')
ax3.tick_params(colors='#8b949e'); ax3.set_facecolor('#161b22')

# 4. Carbs vs Insulin Dose
ax4 = fig.add_subplot(gs[1, 0])
scatter = ax4.scatter(df['carbs_g'], df['insulin_dose'],
                      c=df['activity_level'], cmap='cool', alpha=0.4, s=8)
ax4.set_xlabel('Carbs (g)', color='#8b949e'); ax4.set_ylabel('Insulin Dose (u)', color='#8b949e')
ax4.set_title('Carbs vs Insulin (color=activity)', color='white', fontsize=10)
ax4.tick_params(colors='#8b949e'); ax4.set_facecolor('#161b22')
plt.colorbar(scatter, ax=ax4, label='Activity Level')

# 5. BG vs Post-meal BG
ax5 = fig.add_subplot(gs[1, 1])
ax5.scatter(df['current_bg'], df['bg_after_2h'],
            c=df['insulin_dose'], cmap='plasma', alpha=0.3, s=8)
ax5.set_xlabel('Pre-meal BG (mg/dL)', color='#8b949e')
ax5.set_ylabel('BG after 2h (mg/dL)', color='#8b949e')
ax5.set_title('Pre-meal vs Post-meal BG', color='white', fontsize=10)
ax5.tick_params(colors='#8b949e'); ax5.set_facecolor('#161b22')

# 6. Correlation heatmap
ax6 = fig.add_subplot(gs[1, 2])
corr = df.corr(numeric_only=True)[['insulin_dose','bg_after_2h','hypo_risk']].drop(['insulin_dose','bg_after_2h','hypo_risk'])
im = ax6.imshow(corr.values, cmap='RdYlGn', vmin=-1, vmax=1, aspect='auto')
ax6.set_xticks([0,1,2]); ax6.set_xticklabels(['dose','bg_2h','hypo'], color='white', fontsize=8)
ax6.set_yticks(range(len(corr.index))); ax6.set_yticklabels(corr.index, color='white', fontsize=8)
ax6.set_title('Feature Correlations', color='white', fontsize=10)
for i in range(len(corr.index)):
    for j in range(3):
        ax6.text(j, i, f'{corr.values[i,j]:.2f}', ha='center', va='center', fontsize=8, color='black')

plt.suptitle('Diabetes Dataset — Exploratory Data Analysis', color='white', fontsize=14, fontweight='bold', y=1.01)
plt.savefig('eda.png', dpi=120, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print("EDA plot saved.")

## 3️⃣ Feature Engineering

Adding domain-informed interaction and time features.

In [ ]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    
    # ── Interaction features
    df['carbs_per_unit_weight'] = df['carbs_g'] / df['weight_kg']        # relative meal size
    df['bg_excess']             = (df['current_bg'] - df['target_bg']).clip(0)   # correction needed
    df['effective_icr']         = df['icr'] * (1 - df['activity_level'] * 0.08)  # activity-adjusted ICR
    df['formula_dose']          = df['carbs_g'] / df['icr'] + df['bg_excess'] / df['cf']  # textbook formula

    # ── Time features (dawn phenomenon)
    df['is_dawn']     = ((df['time_of_day'] >= 4) & (df['time_of_day'] <= 9)).astype(int)
    df['sin_time']    = np.sin(2 * np.pi * df['time_of_day'] / 24)  # circular encoding
    df['cos_time']    = np.cos(2 * np.pi * df['time_of_day'] / 24)
    
    # ── BG severity buckets
    df['bg_low']  = (df['current_bg'] < 70).astype(int)
    df['bg_high'] = (df['current_bg'] > 180).astype(int)
    
    return df

df_eng = engineer_features(df)
print(f"Original features : {df.shape[1]}")
print(f"Engineered features: {df_eng.shape[1]}")
print(f"\nNew columns: {list(set(df_eng.columns) - set(df.columns))}")
df_eng.head(3)

## 4️⃣ Model 1 — Insulin Dose Prediction (Regression)

Comparing **Ridge**, **Random Forest**, **Gradient Boosting**, and **XGBoost**.  
Target: continuous units of insulin (e.g. 4.5 units).

In [ ]:
FEATURES_DOSE = [
    'carbs_g', 'current_bg', 'time_of_day', 'activity_level',
    'weight_kg', 'icr', 'cf', 'target_bg',
    'carbs_per_unit_weight', 'bg_excess', 'effective_icr',
    'formula_dose', 'is_dawn', 'sin_time', 'cos_time', 'bg_low', 'bg_high'
]

X = df_eng[FEATURES_DOSE]
y_dose = df_eng['insulin_dose']

X_train, X_test, y_train, y_test = train_test_split(X, y_dose, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")

# ── Define competing models
models_dose = {
    'Ridge': Pipeline([('scaler', StandardScaler()), ('m', Ridge(alpha=1.0))]),
    'RandomForest': Pipeline([('scaler', StandardScaler()),
                               ('m', RandomForestRegressor(n_estimators=300, max_depth=12,
                                                            min_samples_split=5, n_jobs=-1, random_state=42))]),
    'GradientBoosting': Pipeline([('scaler', StandardScaler()),
                                   ('m', GradientBoostingRegressor(n_estimators=300, learning_rate=0.05,
                                                                    max_depth=5, random_state=42))]),
}

if HAS_XGB:
    models_dose['XGBoost'] = Pipeline([('scaler', StandardScaler()),
                                        ('m', XGBRegressor(n_estimators=300, learning_rate=0.05,
                                                           max_depth=5, random_state=42, verbosity=0))])

# ── Cross-validation comparison
print("\n=== 5-Fold Cross-Validation (MAE) ===")
cv_results_dose = {}
for name, pipe in models_dose.items():
    scores = cross_val_score(pipe, X_train, y_train, cv=5,
                              scoring='neg_mean_absolute_error', n_jobs=-1)
    cv_results_dose[name] = -scores
    print(f"  {name:<20} MAE = {-scores.mean():.4f} ± {scores.std():.4f}")

In [ ]:
# ── Train all models & collect test metrics
results_dose = {}
for name, pipe in models_dose.items():
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    results_dose[name] = {
        'MAE':  mean_absolute_error(y_test, preds),
        'RMSE': np.sqrt(mean_squared_error(y_test, preds)),
        'R2':   r2_score(y_test, preds),
        'preds': preds
    }

print("=== Test Set Performance ===")
print(f"{'Model':<22} {'MAE':>8} {'RMSE':>8} {'R²':>8}")
print("-" * 50)
for name, m in results_dose.items():
    print(f"{name:<22} {m['MAE']:>8.4f} {m['RMSE']:>8.4f} {m['R2']:>8.4f}")

best_dose_name = min(results_dose, key=lambda k: results_dose[k]['MAE'])
print(f"\n🏆 Best model: {best_dose_name} (MAE = {results_dose[best_dose_name]['MAE']:.4f})")
best_dose_model = models_dose[best_dose_name]

In [ ]:
# ── Visual: Actual vs Predicted + Residuals
best_preds = results_dose[best_dose_name]['preds']
residuals  = y_test.values - best_preds

fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor='#0d1117')

# Actual vs Predicted
ax = axes[0]; ax.set_facecolor('#161b22')
lim = [0, y_test.max() + 2]
ax.scatter(y_test, best_preds, alpha=0.3, s=8, color=ACCENT)
ax.plot(lim, lim, '--', color=WARN, linewidth=1.5, label='Perfect')
ax.set_xlabel('Actual Dose (u)', color='#8b949e')
ax.set_ylabel('Predicted Dose (u)', color='#8b949e')
ax.set_title(f'{best_dose_name}\nActual vs Predicted', color=ACCENT, fontsize=11)
ax.tick_params(colors='#8b949e'); ax.legend()

# Residuals histogram
ax = axes[1]; ax.set_facecolor('#161b22')
ax.hist(residuals, bins=60, color=GOLD, alpha=0.8, edgecolor='none')
ax.axvline(0, color=WARN, linewidth=1.5)
ax.set_xlabel('Residual (u)', color='#8b949e')
ax.set_ylabel('Count', color='#8b949e')
ax.set_title('Residual Distribution', color=GOLD, fontsize=11)
ax.tick_params(colors='#8b949e')
ax.text(0.05, 0.92, f'Mean: {residuals.mean():.3f}\nStd: {residuals.std():.3f}',
        transform=ax.transAxes, color='white', fontsize=9)

# Model comparison bar
ax = axes[2]; ax.set_facecolor('#161b22')
names = list(results_dose.keys())
maes  = [results_dose[n]['MAE'] for n in names]
colors_bar = [ACCENT if n == best_dose_name else '#30363d' for n in names]
bars = ax.barh(names, maes, color=colors_bar, alpha=0.9)
for bar, val in zip(bars, maes):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', color='white', fontsize=9)
ax.set_xlabel('MAE (units)', color='#8b949e')
ax.set_title('Model MAE Comparison', color='white', fontsize=11)
ax.tick_params(colors='#8b949e')

plt.suptitle('Model 1: Insulin Dose Regression', color='white', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('dose_results.png', dpi=120, bbox_inches='tight', facecolor='#0d1117')
plt.show()

In [ ]:
# ── Feature Importance (from RF)
rf_pipe = models_dose['RandomForest']
rf_model = rf_pipe.named_steps['m']
importances = rf_model.feature_importances_
feat_imp = sorted(zip(FEATURES_DOSE, importances), key=lambda x: -x[1])

fig, ax = plt.subplots(figsize=(10, 6), facecolor='#0d1117')
ax.set_facecolor('#161b22')
names_imp  = [f for f, _ in feat_imp]
vals_imp   = [v for _, v in feat_imp]
colors_imp = [ACCENT if v > 0.05 else '#30363d' for v in vals_imp]
bars = ax.barh(names_imp[::-1], vals_imp[::-1], color=colors_imp[::-1], alpha=0.85)
ax.set_xlabel('Importance', color='#8b949e')
ax.set_title('Random Forest Feature Importances — Insulin Dose Model', color=ACCENT, fontsize=12)
ax.tick_params(colors='#8b949e')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120, bbox_inches='tight', facecolor='#0d1117')
plt.show()

print("Top 5 features:")
for name, val in feat_imp[:5]:
    bar = '█' * int(val * 60)
    print(f"  {name:<25} {bar} {val:.4f}")

## 5️⃣ Hyperparameter Tuning — RandomizedSearchCV

Tuning the best dose model with `RandomizedSearchCV` (more efficient than GridSearch for many params).

In [ ]:
from scipy.stats import randint, uniform

param_dist = {
    'm__n_estimators':     randint(100, 500),
    'm__max_depth':        randint(5, 20),
    'm__min_samples_split': randint(2, 15),
    'm__min_samples_leaf':  randint(1, 8),
    'm__max_features':     ['sqrt', 'log2', 0.5, 0.7],
}

tuning_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('m', RandomForestRegressor(n_jobs=-1, random_state=42))
])

search = RandomizedSearchCV(
    tuning_pipe, param_dist,
    n_iter=30,              # 30 random configurations
    cv=5,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

print("Running RandomizedSearchCV (30 iterations × 5 folds = 150 fits)...")
search.fit(X_train, y_train)

print(f"\n✅ Best CV MAE  : {-search.best_score_:.4f}")
print(f"Best params:")
for k, v in search.best_params_.items():
    print(f"  {k}: {v}")

tuned_preds = search.best_estimator_.predict(X_test)
print(f"\nTest MAE (tuned) : {mean_absolute_error(y_test, tuned_preds):.4f}")
print(f"Test R²  (tuned) : {r2_score(y_test, tuned_preds):.4f}")

best_dose_model = search.best_estimator_   # use tuned model going forward

## 6️⃣ Model 2 — Post-Meal Blood Glucose Prediction (Regression)

Predicts blood glucose level **2 hours after a meal** given the meal composition,  
current BG, insulin administered, and patient parameters.

In [ ]:
y_bg = df_eng['bg_after_2h']

Xb_train, Xb_test, yb_train, yb_test = train_test_split(
    X, y_bg, test_size=0.2, random_state=42)

bg_model = Pipeline([
    ('scaler', StandardScaler()),
    ('m', GradientBoostingRegressor(
        n_estimators=400,
        learning_rate=0.04,
        max_depth=6,
        subsample=0.8,
        min_samples_leaf=5,
        random_state=42
    ))
])

bg_model.fit(Xb_train, yb_train)
yb_pred = bg_model.predict(Xb_test)

print("=== Model 2: Post-Meal BG Prediction ===")
print(f"  MAE  : {mean_absolute_error(yb_test, yb_pred):.2f} mg/dL")
print(f"  RMSE : {np.sqrt(mean_squared_error(yb_test, yb_pred)):.2f} mg/dL")
print(f"  R²   : {r2_score(yb_test, yb_pred):.4f}")

# ── BG zone accuracy (how often do we predict the right clinical zone?)
def bg_zone(bg):
    return np.where(bg < 70, 'Hypo', np.where(bg <= 180, 'Normal', 'Hyper'))

zone_actual = bg_zone(yb_test.values)
zone_pred   = bg_zone(yb_pred)
zone_acc    = (zone_actual == zone_pred).mean()
print(f"\n  Clinical Zone Accuracy: {zone_acc:.4f} ({zone_acc*100:.1f}%)")
print("  (Correctly classifying Hypo/Normal/Hyper)")

In [ ]:
# ── Visualize BG prediction quality
fig, axes = plt.subplots(1, 2, figsize=(13, 5), facecolor='#0d1117')

# Actual vs predicted with clinical zones
ax = axes[0]; ax.set_facecolor('#161b22')
c = np.where(yb_test.values < 70, WARN, np.where(yb_test.values <= 180, ACCENT, GOLD))
ax.scatter(yb_test, yb_pred, c=c, alpha=0.35, s=8)
ax.plot([40, 500], [40, 500], '--', color='white', linewidth=1, alpha=0.5)
ax.axhline(70,  color=WARN, linestyle=':', alpha=0.5, linewidth=1)
ax.axhline(180, color=GOLD, linestyle=':', alpha=0.5, linewidth=1)
ax.set_xlabel('Actual BG (mg/dL)', color='#8b949e')
ax.set_ylabel('Predicted BG (mg/dL)', color='#8b949e')
ax.set_title('Post-Meal BG: Actual vs Predicted\n(red=hypo, green=normal, gold=hyper)', color='white', fontsize=10)
ax.tick_params(colors='#8b949e')

# Error by BG zone
ax = axes[1]; ax.set_facecolor('#161b22')
errors = np.abs(yb_test.values - yb_pred)
zones  = bg_zone(yb_test.values)
for zone, col in [('Hypo', WARN), ('Normal', ACCENT), ('Hyper', GOLD)]:
    zone_errors = errors[zones == zone]
    ax.hist(zone_errors, bins=30, alpha=0.6, color=col, label=f'{zone} (n={len(zone_errors)})', edgecolor='none')
ax.set_xlabel('Absolute Error (mg/dL)', color='#8b949e')
ax.set_ylabel('Count', color='#8b949e')
ax.set_title('Prediction Error by Clinical Zone', color='white', fontsize=10)
ax.legend(fontsize=9); ax.tick_params(colors='#8b949e')

plt.suptitle('Model 2: Post-Meal BG Prediction', color='white', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('bg_prediction.png', dpi=120, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 7️⃣ Model 3 — Hypoglycemia Risk Classifier

**Multi-class classification**: Low / Medium / High hypoglycemia risk.  
Key metrics: accuracy, macro F1, AUC-ROC (one-vs-rest).

In [ ]:
y_hypo = df_eng['hypo_risk']
RISK_LABELS = ['Low', 'Medium', 'High']

Xh_train, Xh_test, yh_train, yh_test = train_test_split(
    X, y_hypo, test_size=0.2, stratify=y_hypo, random_state=42)

print("Class distribution (train):")
for cls, name in enumerate(RISK_LABELS):
    n = (yh_train == cls).sum()
    print(f"  {name}: {n} ({n/len(yh_train)*100:.1f}%)")

hypo_model = Pipeline([
    ('scaler', StandardScaler()),
    ('m', RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        class_weight='balanced',  # handle class imbalance
        min_samples_split=5,
        n_jobs=-1,
        random_state=42
    ))
])

hypo_model.fit(Xh_train, yh_train)
yh_pred      = hypo_model.predict(Xh_test)
yh_proba     = hypo_model.predict_proba(Xh_test)

print(f"\n=== Model 3: Hypoglycemia Risk Classifier ===")
print(f"  Accuracy : {accuracy_score(yh_test, yh_pred):.4f}")
print(f"\n{classification_report(yh_test, yh_pred, target_names=RISK_LABELS)}")

# AUC-ROC (one-vs-rest)
auc_ovr = roc_auc_score(yh_test, yh_proba, multi_class='ovr', average='macro')
print(f"  Macro AUC-ROC (OvR): {auc_ovr:.4f}")

In [ ]:
# ── Confusion matrix + ROC curves
fig, axes = plt.subplots(1, 2, figsize=(13, 5), facecolor='#0d1117')

# Confusion matrix
ax = axes[0]; ax.set_facecolor('#161b22')
cm = confusion_matrix(yh_test, yh_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
im = ax.imshow(cm_norm, cmap='Greens', vmin=0, vmax=1)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f'{cm[i,j]}\n({cm_norm[i,j]:.2f})',
                ha='center', va='center', color='black' if cm_norm[i,j]>0.5 else 'white', fontsize=11)
ax.set_xticks([0,1,2]); ax.set_xticklabels(RISK_LABELS, color='white')
ax.set_yticks([0,1,2]); ax.set_yticklabels(RISK_LABELS, color='white')
ax.set_xlabel('Predicted', color='#8b949e')
ax.set_ylabel('Actual', color='#8b949e')
ax.set_title('Confusion Matrix (normalized)', color=ACCENT, fontsize=11)
plt.colorbar(im, ax=ax)

# ROC curves (one-vs-rest)
ax = axes[1]; ax.set_facecolor('#161b22')
for cls, (name, col) in enumerate(zip(RISK_LABELS, [ACCENT, GOLD, WARN])):
    fpr, tpr, _ = roc_curve((yh_test == cls).astype(int), yh_proba[:, cls])
    auc = roc_auc_score((yh_test == cls).astype(int), yh_proba[:, cls])
    ax.plot(fpr, tpr, color=col, linewidth=2, label=f'{name} (AUC={auc:.3f})')
ax.plot([0,1],[0,1],'--', color='#444', linewidth=1)
ax.set_xlabel('False Positive Rate', color='#8b949e')
ax.set_ylabel('True Positive Rate', color='#8b949e')
ax.set_title('ROC Curves (One-vs-Rest)', color='white', fontsize=11)
ax.legend(fontsize=9); ax.tick_params(colors='#8b949e')

plt.suptitle('Model 3: Hypoglycemia Risk Classifier', color='white', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('hypo_classifier.png', dpi=120, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 8️⃣ Model Explainability — SHAP Values

SHAP (SHapley Additive exPlanations) tells us **why** the model made each prediction —  
essential for medical AI transparency and regulatory compliance.

> Install with: `pip install shap`

In [ ]:
if HAS_SHAP:
    # Use RF from the tuned dose model (unwrap pipeline)
    rf_fitted = best_dose_model.named_steps['m']
    scaler    = best_dose_model.named_steps['scaler']
    X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=FEATURES_DOSE)

    explainer   = shap.TreeExplainer(rf_fitted)
    shap_values = explainer.shap_values(X_test_scaled.head(200))

    # Summary plot
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, X_test_scaled.head(200), show=False,
                      plot_type='violin', color_bar_label='Feature value')
    plt.title('SHAP Summary — Insulin Dose Model', fontsize=12)
    plt.tight_layout()
    plt.savefig('shap_summary.png', dpi=120, bbox_inches='tight')
    plt.show()

    # Force plot for single prediction
    print("\nSHAP explanation for sample patient #0:")
    shap.initjs()
    shap.force_plot(explainer.expected_value, shap_values[0], X_test_scaled.iloc[0],
                    feature_names=FEATURES_DOSE, matplotlib=True, figsize=(14, 3))
    plt.savefig('shap_force.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print("Install SHAP with: pip install shap")
    print("\nAlternative: Permutation Importance (model-agnostic)")
    result = permutation_importance(best_dose_model, X_test, y_test,
                                    n_repeats=20, random_state=42, n_jobs=-1)
    sorted_idx = result.importances_mean.argsort()[::-1]
    print(f"\n{'Feature':<25} {'Mean Decrease':<15} {'Std'}")
    print("-"*50)
    for i in sorted_idx[:10]:
        print(f"{FEATURES_DOSE[i]:<25} {result.importances_mean[i]:>10.4f}    ±{result.importances_std[i]:.4f}")

## 9️⃣ Save Models & Run Inference

Serialize all models with `joblib` for deployment in `app.py`.

In [ ]:
os.makedirs('model', exist_ok=True)

joblib.dump(best_dose_model, 'model/insulin_dose_model.pkl')
joblib.dump(bg_model,        'model/bg_prediction_model.pkl')
joblib.dump(hypo_model,      'model/hypo_risk_model.pkl')
joblib.dump(FEATURES_DOSE,   'model/feature_names.pkl')

print("✅ Models saved:")
for f in os.listdir('model'):
    size = os.path.getsize(f'model/{f}') / 1024
    print(f"   model/{f}  ({size:.0f} KB)")

In [ ]:
# ── Single-patient inference example
def predict_patient(
    carbs_g: float, current_bg: float, time_of_day: int,
    activity_level: int, weight_kg: float, icr: float,
    cf: float, target_bg: float
) -> dict:
    """
    Run all 3 models for a single patient input.
    Returns insulin dose, predicted BG after 2h, and hypo risk.
    """
    raw = pd.DataFrame([{
        'carbs_g': carbs_g, 'current_bg': current_bg,
        'time_of_day': time_of_day, 'activity_level': activity_level,
        'weight_kg': weight_kg, 'icr': icr, 'cf': cf, 'target_bg': target_bg
    }])
    eng = engineer_features(raw)[FEATURES_DOSE]

    dose     = best_dose_model.predict(eng)[0]
    bg_2h    = bg_model.predict(eng)[0]
    risk_id  = hypo_model.predict(eng)[0]
    risk_prb = hypo_model.predict_proba(eng)[0]

    return {
        'insulin_dose':  round(dose, 2),
        'bg_after_2h':   round(bg_2h, 1),
        'hypo_risk':     RISK_LABELS[risk_id],
        'risk_proba':    dict(zip(RISK_LABELS, risk_prb.round(3)))
    }

# ── Example patients
patients = [
    dict(carbs_g=60, current_bg=180, time_of_day=12, activity_level=1,
         weight_kg=70, icr=10, cf=50, target_bg=100),
    dict(carbs_g=20, current_bg=95,  time_of_day=6,  activity_level=2,
         weight_kg=65, icr=12, cf=60, target_bg=100),
    dict(carbs_g=120, current_bg=280, time_of_day=19, activity_level=0,
         weight_kg=90, icr=8, cf=40, target_bg=110),
]

print(f"{'':=<60}")
for i, p in enumerate(patients):
    result = predict_patient(**p)
    print(f"\nPatient {i+1}:")
    print(f"  Input : {p['carbs_g']}g carbs | BG {p['current_bg']} mg/dL | Activity {p['activity_level']}")
    print(f"  Dose  : {result['insulin_dose']} units")
    print(f"  BG 2h : {result['bg_after_2h']} mg/dL")
    print(f"  Risk  : {result['hypo_risk']}  {result['risk_proba']}")
print(f"{'':=<60}")

## 🚀 Next Steps & Real-World Extensions

### Use Real Data
- **OhioT1DM** — 12 T1D patients, 8 weeks of CGM + insulin pump + meals
- **Pima Indians Diabetes (Kaggle)** — classification baseline
- **NHANES** — large population health survey

### Advanced Models to Try
```python
# LSTM for time-series BG prediction
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

# Bayesian Neural Network for uncertainty quantification
# → critical for medical AI: "I'm 70% confident, ±2 units"

# Reinforcement Learning for closed-loop insulin delivery
# → artificial pancreas systems use RL
```

### Model Improvements
- **Temporal features**: rolling CGM trend (rising/falling BG changes dose)
- **IOB (Insulin on Board)**: remaining active insulin from previous doses
- **Meal glycemic index**: refined vs complex carbs absorb differently
- **Patient personalization**: fine-tune on individual's historical data

### Deployment
```bash
python app.py          # Flask REST API
# → POST /predict with JSON → returns dose, BG, risk
```

### GitHub Structure
```
diabetes-ml/
├── data/generate_data.py
├── model/                    ← .pkl files
├── notebooks/
│   └── diabetes_ml.ipynb     ← this notebook
├── app.py                    ← Flask API
├── requirements.txt
└── README.md
```

---
> ⚠️ **Disclaimer**: This is an educational ML project. Real insulin dosing decisions must involve certified medical devices and endocrinologist oversight.